<a href="https://colab.research.google.com/github/oppheonix53-maker/liveclip/blob/main/liveclip_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📱 liveclip on your phone (free)

Turn any GTA stream into ready-to-post vertical clips: **streamer's face on top, gameplay below, captions on.** It runs on Google's free computers, so you only need a phone and a Google account. No laptop, no install.

**How to use it**
1. Paste a YouTube or Twitch link in the box below. It can be a **live stream** or a **finished video**.
2. Tap the **▶ play button**. If Colab says *"This notebook was not authored by Google"*, tap **Run anyway**.
3. Keep this tab open. Your clips show up below with a **Download** button.

**On an iPhone?** iPhone browsers can't save videos from this page. Tick **save_to_google_drive** before you tap play: every clip also goes to your Google Drive (My Drive → liveclip). Then in the Google Drive app: ⋯ → **Send a copy** → **Save Video**.

Live stream: it clips the moments when the chat explodes. Finished video: it finds the loudest moments (shouting, hype, big reactions). It finds the streamer's facecam by itself; if it picks the wrong spot, choose the corner in **facecam**.

> **Only clip streamers who allow clipping** (or who run an official clipping program). Clips without permission get copyright strikes and no payout.

Made by [@aaditya.knows](https://instagram.com/aaditya.knows) · code: [github.com/Aadi1659/liveclip](https://github.com/Aadi1659/liveclip)

In [ ]:
#@title ▶ LiveClip AI — Paste Link & Tap Play
link = ""  #@param {type:"string"}
mode = "auto"  #@param ["auto", "live", "recorded"]
facecam = "find it for me"  #@param ["find it for me", "top left", "top right", "bottom left", "bottom right", "no facecam"]
captions = "hinglish"  #@param ["hinglish", "english", "hi", "auto", "off"]
clips = 5  #@param {type:"slider", min:1, max:15, step:1}
clip_length_sec = 30  #@param {type:"slider", min:15, max:60, step:5}
sensitivity = "medium"  #@param ["high (more clips)", "medium", "low (only huge hype)"]
watch_minutes = 60  #@param {type:"slider", min:10, max:180, step:10}
save_to_google_drive = True  #@param {type:"boolean"}
#@markdown 📱 **iPhone / Mobile users:** Tick `save_to_google_drive` so clips save directly to Drive instead of relying on mobile browser downloads.

import os, re, signal, subprocess, sys, shutil, zipfile, datetime, urllib.request
from IPython.display import display, HTML, Video
import ipywidgets as widgets
from google.colab import files

# --- 1. Validate URL & Hardware ---
url = link.strip()
if not re.search(r"(youtube\.com|youtu\.be|twitch\.tv|kick\.com)/", url):
    raise SystemExit("❌ Paste a valid YouTube, Twitch, or Kick link in the 'link' box first.")

has_gpu = subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not has_gpu:
    display(HTML(
        '<div style="padding:10px 14px;border-radius:8px;background:#fff3cd;color:#664d03;font:14px sans-serif;margin-bottom:10px">'
        '⚠️ <b>No GPU detected.</b> AI analysis, Whisper, and rendering will be slow. '
        'Go to <b>Runtime → Change runtime type → T4 GPU</b> for 5–10× faster clipping.</div>'
    ))

# --- 2. Setup / Update Your Forked Repo ---
REPO_URL = "https://github.com/oppheonix53-maker/liveclip.git"
REPO_DIR = "/content/liveclip"

if not os.path.exists(f"{REPO_DIR}/liveclip.py"):
    print("⏳ First-time setup (about 1 minute)…")
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "streamlink>=7.0", "faster-whisper>=1.0", "transformers", "yt-dlp", "opencv-python-headless<5"
    ], check=True)
    print("✅ Setup complete.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], stderr=subprocess.DEVNULL)

# Ensure Montserrat-ExtraBold.ttf is available for viral .ass captions
FONT_DIR = f"{REPO_DIR}/assets/fonts"
FONT_PATH = f"{FONT_DIR}/Montserrat-ExtraBold.ttf"
if not os.path.exists(FONT_PATH):
    os.makedirs(FONT_DIR, exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-ExtraBold.ttf",
            FONT_PATH
        )
    except Exception:
        pass

# --- 3. Output & Google Drive Setup ---
run_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
out_dir = f"/content/clips/{run_stamp}"
os.makedirs(out_dir, exist_ok=True)

drive_dir = None
if save_to_google_drive:
    from google.colab import drive
    print("📁 Connecting Google Drive…")
    drive.mount("/content/drive", force_remount=False)
    drive_dir = f"/content/drive/MyDrive/liveclip/{run_stamp}"
    os.makedirs(drive_dir, exist_ok=True)

# --- 4. Build Command ---
face_arg = {"find it for me": "auto", "no facecam": "none"}.get(facecam, facecam.replace(" ", "-"))
spike_arg = {"high (more clips)": "1.4", "medium": "1.7", "low (only huge hype)": "2.1"}[sensitivity]
post_sec = max(5, int(clip_length_sec * 0.2))
pre_sec = clip_length_sec - post_sec

cmd = [
    sys.executable, "-u", f"{REPO_DIR}/liveclip.py", url,
    "--mode", mode,
    "--face", face_arg,
    "--out", out_dir,
    "--pre", str(pre_sec),
    "--post", str(post_sec),
    "--spike", spike_arg,
    "--duration", str(watch_minutes * 60),
    "--max-clips", str(clips),
    "--captions", captions,
    "--model", "auto",
    "--device", "cuda" if has_gpu else "cpu",
]

# --- 5. Memory-Safe Clip Renderer ---
saved_clips = []

def render_clip(path):
    """Display video preview + native Colab download button without crashing browser RAM."""
    saved_clips.append(path)
    name = os.path.basename(path)
    size_mb = os.path.getsize(path) / (1024 * 1024)

    display(HTML(f'<p style="font:600 15px sans-serif;margin:16px 0 6px">🎬 {name} <span style="font-weight:400;color:#666">({size_mb:.1f} MB)</span></p>'))
    display(Video(path, embed=True, width=340, html_attributes="controls playsinline style='border-radius:12px'"))

    btn = widgets.Button(
        description=f"⬇ Download {name}",
        button_style="warning",
        layout=widgets.Layout(width="auto", margin="6px 0 4px 0")
    )
    btn.on_click(lambda _, p=path: files.download(p))
    display(btn)

    if drive_dir:
        shutil.copy2(path, drive_dir)
        display(HTML(
            f'<p style="font:13px sans-serif;color:#1e7e34;margin:4px 0 10px">'
            f'✅ Saved to Drive → <code>My Drive/liveclip/{run_stamp}/{name}</code></p>'
        ))

# --- 6. Run LiveClip ---
print(f"🚀 Starting LiveClip AI ({mode} mode, max {clips} clips). Tap ■ to stop gracefully.\n")
env = dict(os.environ, HF_HUB_VERBOSITY="error", HF_HUB_DISABLE_PROGRESS_BARS="1", PYTHONWARNINGS="ignore")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, start_new_session=True)

stopping = False
while True:
    try:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            continue
        line = line.rstrip()
        if any(noise in line for noise in ("unauthenticated requests", "[ctranslate2]")):
            continue
        print(line)
        m = re.search(r"saved (\S+\.mp4)$", line)
        if m and os.path.exists(m.group(1)):
            render_clip(m.group(1))
    except KeyboardInterrupt:
        if not stopping:
            stopping = True
            print("\n🛑 Stopping… finishing any clips currently rendering.")
            proc.send_signal(signal.SIGINT)
        else:
            proc.kill()
            break

# --- 7. Summary & Batch ZIP Download ---
if proc.returncode not in (0, None, -signal.SIGINT):
    print(f"\n⚠️ liveclip exited with code {proc.returncode}. Check if the link is public and not geo-blocked.")

if saved_clips:
    print(f"\n🎉 Done! Generated {len(saved_clips)} clip(s).")
    def download_zip(_):
        zip_path = f"/content/liveclip_{run_stamp}.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for c in saved_clips:
                zf.write(c, os.path.basename(c))
        files.download(zip_path)

    zip_btn = widgets.Button(description=f"📦 Download All {len(saved_clips)} Clips (.zip)", button_style="success")
    zip_btn.on_click(download_zip)
    display(zip_btn)
else:
    print("\nℹ️ No clips were generated. Try lowering sensitivity to 'high (more clips)' or checking the URL.")

In [ ]:
#@title 🎞 Show all my clips again
import glob, os, base64
from IPython.display import display, HTML
clips = sorted(glob.glob("/content/clips/*.mp4"))
if not clips: print("No clips yet. Run the cell above first.")
for path in clips:
    b64 = base64.b64encode(open(path, "rb").read()).decode(); name = os.path.basename(path)
    display(HTML(f'<p style="font:600 15px sans-serif;margin:14px 0 6px">🎬 {name}</p>'
                 f'<video controls playsinline style="width:100%;max-width:360px;border-radius:12px" src="data:video/mp4;base64,{b64}"></video>'
                 f'<br><a download="{name}" href="data:video/mp4;base64,{b64}" style="display:inline-block;margin-top:6px;'
                 f'padding:10px 16px;border-radius:10px;background:#e8684a;color:#fff;font:600 15px sans-serif;text-decoration:none">⬇ Download {name}</a>'))

### Tips
- **Wrong part shown as the face?** Pick the corner in **facecam** and tap play again. Streams with no facecam: pick *no facecam*.
- **Live stream, no clips after 10 minutes?** The chat hasn't exploded yet. Keep it running, or paste a finished video instead.
- **Want it faster?** Menu → **Runtime → Change runtime type → T4 GPU** before you tap play. Captions then take a couple of seconds instead of ~30.
- **Phone screen locked or tab closed?** The clips stay on the Colab computer for a while. Come back and run *Show all my clips again*. With the Google Drive box ticked they're also safe in your Drive.
- **Download button does nothing on iPhone?** That's an iPhone limit. Tick **save_to_google_drive** and run again.
- Watch each clip before you post it. Captions can have mistakes; fix them in Edits or CapCut.
- Free Colab sessions stop after a few hours or when idle. That's Google's limit, not this tool's.